# Tema 4 - Par Diferencial
**Electronica General - 2o GIERM**

El par diferencial es el bloque fundamental de los amplificadores operacionales y de la mayoria de circuitos analogicos integrados. Permite amplificar la diferencia entre dos senales rechazando las componentes comunes (ruido, interferencias).

## Indice

1. [Introduccion y motivacion](#1-introduccion)
2. [Circuito basico del par diferencial MOS](#2-circuito-mos)
3. [Analisis de gran senal: curva de transferencia](#3-gran-senal)
4. [Analisis de pequena senal](#4-pequena-senal)
5. [Ganancia modo comun y CMRR](#5-cmrr)
6. [Carga activa: espejo de corriente](#6-carga-activa)
7. [CMRR en frecuencia](#7-cmrr-frecuencia)
8. [Ejercicios resueltos](#8-ejercicios)
9. [Catalogo de problemas tipo](#9-catalogo)
10. [Resumen final](#10-resumen)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch

# Paleta de colores consistente
COLOR_PRINCIPAL = '#2171b5'   # azul - curvas principales
COLOR_RECTA = '#cb181d'       # rojo - rectas de carga, limites
COLOR_PUNTO = '#238b45'       # verde - puntos de operacion
COLOR_N = '#a6cee3'           # azul claro
COLOR_P = '#b2df8a'           # verde claro
COLOR_DESTAQUE = '#ff7f00'    # naranja - destacados

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11,
                     'axes.grid': True, 'grid.alpha': 0.3})
print('Entorno listo.')

## 1. Introduccion y motivacion <a id="1-introduccion"></a>

En cualquier sistema electronico necesitamos amplificar senales debiles en presencia de ruido. El **par diferencial** resuelve esto:

| Propiedad | Descripcion |
|---|---|
| Amplifica $v_{id} = v_1 - v_2$ | La senal util (diferencia) |
| Rechaza $v_{cm} = (v_1+v_2)/2$ | Ruido, offsets, interferencias |
| CMRR alto | Razon de rechazo al modo comun |

Este notebook cubre el par diferencial MOS (NMOS) y menciona las diferencias con BJT cuando es relevante.

## 2. Circuito basico del par diferencial MOS <a id="2-circuito-mos"></a>

El par diferencial NMOS basico consta de:
- Dos transistores M1, M2 emparejados (mismos $W/L$, $V_{th}$, $k_n$)
- Una fuente de corriente $I_{SS}$ en la cola (source comun)
- Resistencias de drenador $R_D$ iguales (carga pasiva)

La corriente de cola se reparte entre M1 y M2 segun la diferencia $v_{id}$.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('Par diferencial NMOS con carga pasiva $R_D$', fontsize=14, fontweight='bold')

# VDD rail
ax.plot([2, 8], [9.5, 9.5], 'k-', lw=2)
ax.text(5, 9.7, '$V_{DD}$', ha='center', fontsize=13, fontweight='bold')

# RD left
ax.plot([3, 3], [9.5, 7.5], color=COLOR_RECTA, lw=6, solid_capstyle='butt')
ax.text(2.3, 8.5, '$R_D$', fontsize=12, color=COLOR_RECTA, fontweight='bold')

# RD right
ax.plot([7, 7], [9.5, 7.5], color=COLOR_RECTA, lw=6, solid_capstyle='butt')
ax.text(7.5, 8.5, '$R_D$', fontsize=12, color=COLOR_RECTA, fontweight='bold')

# Drain wires
ax.plot([3, 3], [7.5, 6.2], 'k-', lw=1.5)
ax.plot([7, 7], [7.5, 6.2], 'k-', lw=1.5)

# Output labels
ax.plot([3, 1.5], [7.0, 7.0], 'k--', lw=1)
ax.text(1.0, 7.0, '$v_{o1}$', fontsize=12, color=COLOR_PRINCIPAL, fontweight='bold')
ax.plot([7, 8.5], [7.0, 7.0], 'k--', lw=1)
ax.text(8.7, 7.0, '$v_{o2}$', fontsize=12, color=COLOR_PRINCIPAL, fontweight='bold')

# M1 transistor (left)
ax.add_patch(plt.Rectangle((2.3, 5.2), 1.4, 1.0, fc=COLOR_N, ec='black', lw=1.5))
ax.text(3, 5.7, 'M1', ha='center', va='center', fontsize=11, fontweight='bold')

# M2 transistor (right)
ax.add_patch(plt.Rectangle((6.3, 5.2), 1.4, 1.0, fc=COLOR_N, ec='black', lw=1.5))
ax.text(7, 5.7, 'M2', ha='center', va='center', fontsize=11, fontweight='bold')

# Gate inputs
ax.plot([2.3, 1.2], [5.7, 5.7], 'k-', lw=1.5)
ax.text(0.5, 5.7, '$v_{G1}$', fontsize=12, color=COLOR_PUNTO, fontweight='bold')
ax.plot([7.7, 8.8], [5.7, 5.7], 'k-', lw=1.5)
ax.text(8.9, 5.7, '$v_{G2}$', fontsize=12, color=COLOR_PUNTO, fontweight='bold')

# Source wires to common node
ax.plot([3, 3], [5.2, 4.2], 'k-', lw=1.5)
ax.plot([7, 7], [5.2, 4.2], 'k-', lw=1.5)
ax.plot([3, 7], [4.2, 4.2], 'k-', lw=1.5)
ax.plot([5, 5], [4.2, 3.2], 'k-', lw=1.5)

# VS node label
ax.plot(5, 4.2, 'ko', ms=5)
ax.text(5.4, 4.4, '$V_S$', fontsize=11)

# ISS current source
ax.annotate('', xy=(5, 2.0), xytext=(5, 3.2),
            arrowprops=dict(arrowstyle='->', color=COLOR_DESTAQUE, lw=2.5))
ax.text(5.5, 2.5, '$I_{SS}$', fontsize=13, color=COLOR_DESTAQUE, fontweight='bold')

# Ground
ax.plot([4.5, 5.5], [1.5, 1.5], 'k-', lw=2)
ax.plot([4.7, 5.3], [1.3, 1.3], 'k-', lw=1.5)
ax.plot([4.85, 5.15], [1.1, 1.1], 'k-', lw=1)
ax.plot([5, 5], [2.0, 1.5], 'k-', lw=1.5)

plt.tight_layout()
plt.show()

### Definiciones de senales

**Tension diferencial de entrada:**
$$v_{id} = v_{G1} - v_{G2}$$

**Tension de modo comun de entrada:**
$$v_{cm} = \frac{v_{G1} + v_{G2}}{2}$$

De estas se obtiene:
$$v_{G1} = v_{cm} + \frac{v_{id}}{2}, \qquad v_{G2} = v_{cm} - \frac{v_{id}}{2}$$

## 3. Analisis de gran senal: curva de transferencia <a id="3-gran-senal"></a>

Para NMOS en saturacion con $I_D = \frac{k_n}{2}(V_{GS}-V_{th})^2$, el reparto de corriente es:

$$I_{D1} = \frac{I_{SS}}{2} + \frac{k_n}{2} v_{id} \sqrt{\frac{I_{SS}}{k_n} - \frac{v_{id}^2}{4}}$$

$$I_{D2} = I_{SS} - I_{D1}$$

La corriente se satura en $I_{SS}$ cuando $|v_{id}| \geq \sqrt{2I_{SS}/k_n}$.

In [ ]:
# Curva de transferencia del par diferencial MOS
kn = 2e-3   # A/V^2
ISS = 0.4e-3  # A
Vov = np.sqrt(ISS / kn)  # sqrt(ISS/kn)

vid = np.linspace(-0.8, 0.8, 500)

# Calculo de ID1 para MOS
ID1_mos = np.zeros_like(vid)
for i, v in enumerate(vid):
    disc = ISS/kn - v**2/4
    if disc < 0:
        ID1_mos[i] = ISS if v > 0 else 0.0
    else:
        ID1_mos[i] = ISS/2 + (kn/2)*v*np.sqrt(disc)
        ID1_mos[i] = np.clip(ID1_mos[i], 0, ISS)
ID2_mos = ISS - ID1_mos

# Curva BJT (tanh) para comparacion
VT = 26e-3  # tension termica
ID1_bjt = ISS * 1/(1 + np.exp(-vid/VT))
ID2_bjt = ISS - ID1_bjt

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# MOS
ax = axes[0]
ax.plot(vid*1e3, ID1_mos*1e3, color=COLOR_PRINCIPAL, lw=2.5, label='$I_{D1}$')
ax.plot(vid*1e3, ID2_mos*1e3, color=COLOR_RECTA, lw=2.5, ls='--', label='$I_{D2}$')
ax.axhline(ISS*1e3/2, color='gray', ls=':', lw=1)
ax.axvline(0, color='gray', ls=':', lw=1)
ax.set_xlabel('$v_{id}$ (mV)')
ax.set_ylabel('$I_D$ (mA)')
ax.set_title('Par diferencial MOS', fontweight='bold')
ax.legend()
ax.set_ylim(-0.02, ISS*1e3*1.1)
ax.annotate(f'$\\sqrt{{2I_{{SS}}/k_n}}$ = {Vov*np.sqrt(2)*1e3:.0f} mV',
            xy=(Vov*np.sqrt(2)*1e3, ISS*1e3), xytext=(250, 0.3),
            fontsize=10, color=COLOR_PUNTO, fontweight='bold',
            arrowprops=dict(arrowstyle='->', color=COLOR_PUNTO, lw=1.5))

# BJT
ax = axes[1]
ax.plot(vid*1e3, ID1_bjt*1e3, color=COLOR_PRINCIPAL, lw=2.5, label='$I_{C1}$')
ax.plot(vid*1e3, ID2_bjt*1e3, color=COLOR_RECTA, lw=2.5, ls='--', label='$I_{C2}$')
ax.axhline(ISS*1e3/2, color='gray', ls=':', lw=1)
ax.axvline(0, color='gray', ls=':', lw=1)
ax.set_xlabel('$v_{id}$ (mV)')
ax.set_ylabel('$I_C$ (mA)')
ax.set_title('Par diferencial BJT', fontweight='bold')
ax.legend()
ax.set_ylim(-0.02, ISS*1e3*1.1)
ax.annotate(f'Transicion lineal $\\approx$ {4*VT*1e3:.0f} mV',
            xy=(4*VT*1e3, ISS*0.98*1e3), xytext=(250, 0.3),
            fontsize=10, color=COLOR_PUNTO, fontweight='bold',
            arrowprops=dict(arrowstyle='->', color=COLOR_PUNTO, lw=1.5))

plt.suptitle('Curvas de transferencia: MOS vs BJT', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f'MOS: rango lineal = +/-{Vov*np.sqrt(2)*1e3:.1f} mV (mas amplio)')
print(f'BJT: rango lineal ~ +/-{4*VT*1e3:.0f} mV (mas estrecho, mayor gm)')

## 4. Analisis de pequena senal <a id="4-pequena-senal"></a>

Cuando $v_{id}$ es pequeno, el par diferencial se comporta como un amplificador lineal.

### Medio circuito diferencial

Por simetria, si aplicamos solo senal diferencial ($v_{cm} = 0$), el nodo comun (source) es un **tierra virtual**. Cada mitad se analiza como un amplificador source comun:

$$A_d = \frac{v_{o1} - v_{o2}}{v_{id}} = -g_m \cdot R_D$$

Donde $g_m = \sqrt{2 k_n I_{SS}} = \frac{2I_D}{V_{OV}}$ es la transconductancia de cada transistor con $I_D = I_{SS}/2$.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# --- Half-circuit differential ---
ax = axes[0]
ax.set_xlim(0, 6)
ax.set_ylim(0, 7)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('Medio circuito diferencial\n(source = tierra virtual)', fontsize=12, fontweight='bold')

# VDD
ax.plot([3, 3], [6.5, 5.5], 'k-', lw=1.5)
ax.text(3, 6.7, '$V_{DD}$', ha='center', fontsize=11)
# RD
ax.plot([3, 3], [5.5, 4.5], color=COLOR_RECTA, lw=6, solid_capstyle='butt')
ax.text(3.6, 5.0, '$R_D$', fontsize=11, color=COLOR_RECTA, fontweight='bold')
# Wire drain
ax.plot([3, 3], [4.5, 3.8], 'k-', lw=1.5)
ax.text(3.6, 4.1, '$v_o$', fontsize=11, color=COLOR_PRINCIPAL, fontweight='bold')
# Transistor
ax.add_patch(plt.Rectangle((2.3, 2.8), 1.4, 1.0, fc=COLOR_N, ec='black', lw=1.5))
ax.text(3, 3.3, 'M1', ha='center', fontsize=10, fontweight='bold')
# Gate
ax.plot([2.3, 1.0], [3.3, 3.3], 'k-', lw=1.5)
ax.text(0.3, 3.3, '$\\frac{v_{id}}{2}$', fontsize=12, color=COLOR_PUNTO, fontweight='bold')
# Source to ground
ax.plot([3, 3], [2.8, 2.0], 'k-', lw=1.5)
ax.plot([2.5, 3.5], [2.0, 2.0], 'k-', lw=2)
ax.plot([2.65, 3.35], [1.85, 1.85], 'k-', lw=1.5)
ax.plot([2.8, 3.2], [1.7, 1.7], 'k-', lw=1)
ax.text(3, 1.3, 'Tierra virtual', ha='center', fontsize=10, color=COLOR_DESTAQUE, fontstyle='italic')

# --- Half-circuit common-mode ---
ax = axes[1]
ax.set_xlim(0, 6)
ax.set_ylim(0, 7)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('Medio circuito modo comun\n(source con $2R_{SS}$)', fontsize=12, fontweight='bold')

# VDD
ax.plot([3, 3], [6.5, 5.5], 'k-', lw=1.5)
ax.text(3, 6.7, '$V_{DD}$', ha='center', fontsize=11)
# RD
ax.plot([3, 3], [5.5, 4.5], color=COLOR_RECTA, lw=6, solid_capstyle='butt')
ax.text(3.6, 5.0, '$R_D$', fontsize=11, color=COLOR_RECTA, fontweight='bold')
# Wire drain
ax.plot([3, 3], [4.5, 3.8], 'k-', lw=1.5)
ax.text(3.6, 4.1, '$v_o$', fontsize=11, color=COLOR_PRINCIPAL, fontweight='bold')
# Transistor
ax.add_patch(plt.Rectangle((2.3, 2.8), 1.4, 1.0, fc=COLOR_N, ec='black', lw=1.5))
ax.text(3, 3.3, 'M1', ha='center', fontsize=10, fontweight='bold')
# Gate
ax.plot([2.3, 1.0], [3.3, 3.3], 'k-', lw=1.5)
ax.text(0.3, 3.3, '$v_{cm}$', fontsize=12, color=COLOR_PUNTO, fontweight='bold')
# Source to 2RSS
ax.plot([3, 3], [2.8, 2.2], 'k-', lw=1.5)
ax.plot([3, 3], [2.2, 1.2], color=COLOR_DESTAQUE, lw=6, solid_capstyle='butt')
ax.text(3.6, 1.7, '$2R_{SS}$', fontsize=11, color=COLOR_DESTAQUE, fontweight='bold')
# Ground
ax.plot([3, 3], [1.2, 0.6], 'k-', lw=1.5)
ax.plot([2.5, 3.5], [0.6, 0.6], 'k-', lw=2)
ax.plot([2.65, 3.35], [0.45, 0.45], 'k-', lw=1.5)
ax.plot([2.8, 3.2], [0.3, 0.3], 'k-', lw=1)

plt.tight_layout()
plt.show()

### Resumen de ganancias de pequena senal

| Parametro | Formula (carga pasiva $R_D$) | Unidad |
|---|---|---|
| $g_m$ | $\sqrt{2 k_n I_{SS}}$ | A/V |
| $A_d$ (diferencial) | $-g_m R_D$ | V/V |
| $A_{cm}$ (modo comun) | $-\frac{g_m R_D}{1 + 2 g_m R_{SS}}$ | V/V |
| $A_{cm}$ (aprox.) | $\approx -\frac{R_D}{2 R_{SS}}$ si $g_m R_{SS} \gg 1$ | V/V |
| CMRR | $1 + 2 g_m R_{SS} \approx 2 g_m R_{SS}$ | - |

## 5. Ganancia modo comun y CMRR <a id="5-cmrr"></a>

Cuando ambas entradas tienen la misma senal ($v_{id}=0$, $v_{cm}\neq 0$), el nodo source NO es tierra virtual. La corriente por $R_{SS}$ cambia y produce una salida no deseada.

$$A_{cm} = \frac{-g_m R_D}{1 + 2 g_m R_{SS}} \approx \frac{-R_D}{2 R_{SS}}$$

El **CMRR** (Common Mode Rejection Ratio) mide cuanto mejor amplificamos la senal diferencial frente al modo comun:

$$\text{CMRR} = \left|\frac{A_d}{A_{cm}}\right| = 1 + 2 g_m R_{SS} \approx 2 g_m R_{SS}$$

En dB: $\text{CMRR}_{dB} = 20 \log_{10}(\text{CMRR})$

In [ ]:
# Calculo numerico de CMRR para distintos valores de RSS
kn = 2e-3  # A/V^2
ISS = 0.4e-3  # A
RD = 10e3  # 10 kohm
gm = np.sqrt(2 * kn * ISS)  # gm con ID = ISS/2 -> gm = sqrt(2*kn*ISS)

print(f'Parametros del par diferencial MOS:')
print(f'  kn = {kn*1e3:.1f} mA/V^2')
print(f'  ISS = {ISS*1e3:.2f} mA')
print(f'  RD = {RD/1e3:.0f} kohm')
print(f'  gm = {gm*1e3:.2f} mA/V')
print(f'  Ad = -gm*RD = {-gm*RD:.1f} V/V = {20*np.log10(gm*RD):.1f} dB')
print()

RSS_vals = np.array([10e3, 50e3, 100e3, 500e3, 1e6])
print(f'{"RSS (kohm)":>12} {"Acm (V/V)":>12} {"CMRR":>10} {"CMRR (dB)":>12}')
print('-' * 50)
for RSS in RSS_vals:
    Acm = -gm * RD / (1 + 2*gm*RSS)
    CMRR = abs(-gm*RD / Acm)
    CMRR_dB = 20 * np.log10(CMRR)
    print(f'{RSS/1e3:>12.0f} {Acm:>12.4f} {CMRR:>10.0f} {CMRR_dB:>12.1f}')

## 6. Carga activa: espejo de corriente <a id="6-carga-activa"></a>

Sustituir las resistencias $R_D$ por un **espejo de corriente PMOS** (M3-M4) como carga activa mejora drasticamente la ganancia:

- El espejo fuerza $I_{D3} = I_{D4}$
- La salida es **single-ended** (un solo nodo)
- La ganancia diferencial pasa a depender de $r_o$:

$$A_d = g_m (r_{o2} \| r_{o4})$$

Donde $r_{o2}$ y $r_{o4}$ son las resistencias de salida de M2 y M4 respectivamente.

Esto da ganancias de **cientos a miles** de V/V en circuitos integrados.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7.5))
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('Par diferencial con carga activa (espejo PMOS)', fontsize=14, fontweight='bold')

# VDD rail
ax.plot([2, 8], [9.5, 9.5], 'k-', lw=2)
ax.text(5, 9.7, '$V_{DD}$', ha='center', fontsize=13, fontweight='bold')

# M3 PMOS (left)
ax.add_patch(plt.Rectangle((2.3, 7.8), 1.4, 1.0, fc=COLOR_P, ec='black', lw=1.5))
ax.text(3, 8.3, 'M3', ha='center', fontsize=10, fontweight='bold')
ax.plot([3, 3], [9.5, 8.8], 'k-', lw=1.5)  # source to VDD
ax.plot([3, 3], [7.8, 6.8], 'k-', lw=1.5)  # drain down

# M4 PMOS (right)
ax.add_patch(plt.Rectangle((6.3, 7.8), 1.4, 1.0, fc=COLOR_P, ec='black', lw=1.5))
ax.text(7, 8.3, 'M4', ha='center', fontsize=10, fontweight='bold')
ax.plot([7, 7], [9.5, 8.8], 'k-', lw=1.5)  # source to VDD
ax.plot([7, 7], [7.8, 6.8], 'k-', lw=1.5)  # drain down

# Mirror connection (gates tied)
ax.plot([2.3, 2.0], [8.3, 8.3], 'k-', lw=1.5)
ax.plot([2.0, 2.0], [8.3, 7.0], 'k-', lw=1.5)
ax.plot([2.0, 3.0], [7.0, 7.0], 'k-', lw=1.5)  # gate tied to drain M3
ax.plot([2.0, 2.0], [8.3, 8.3], 'k-', lw=1.5)
# Connect M3 gate to M4 gate
ax.plot([2.0, 6.3], [8.3, 8.3], 'k--', lw=1.2, color=COLOR_DESTAQUE)
ax.text(4.5, 8.55, 'espejo', fontsize=10, ha='center', color=COLOR_DESTAQUE, fontstyle='italic')

# Output node
ax.plot([7, 9], [7.0, 7.0], 'k-', lw=1.5)
ax.text(9.1, 7.0, '$v_o$', fontsize=13, color=COLOR_PRINCIPAL, fontweight='bold')
ax.plot(7, 7.0, 'o', color=COLOR_PUNTO, ms=8, zorder=5)

# M1 NMOS (left)
ax.add_patch(plt.Rectangle((2.3, 5.0), 1.4, 1.0, fc=COLOR_N, ec='black', lw=1.5))
ax.text(3, 5.5, 'M1', ha='center', fontsize=10, fontweight='bold')
ax.plot([3, 3], [6.8, 6.0], 'k-', lw=1.5)

# M2 NMOS (right)
ax.add_patch(plt.Rectangle((6.3, 5.0), 1.4, 1.0, fc=COLOR_N, ec='black', lw=1.5))
ax.text(7, 5.5, 'M2', ha='center', fontsize=10, fontweight='bold')
ax.plot([7, 7], [6.8, 6.0], 'k-', lw=1.5)

# Gates
ax.plot([2.3, 1.0], [5.5, 5.5], 'k-', lw=1.5)
ax.text(0.3, 5.5, '$v_{G1}$', fontsize=12, color=COLOR_PUNTO, fontweight='bold')
ax.plot([7.7, 9.0], [5.5, 5.5], 'k-', lw=1.5)
ax.text(9.1, 5.5, '$v_{G2}$', fontsize=12, color=COLOR_PUNTO, fontweight='bold')

# Source wires to common node
ax.plot([3, 3], [5.0, 4.0], 'k-', lw=1.5)
ax.plot([7, 7], [5.0, 4.0], 'k-', lw=1.5)
ax.plot([3, 7], [4.0, 4.0], 'k-', lw=1.5)
ax.plot([5, 5], [4.0, 3.0], 'k-', lw=1.5)
ax.plot(5, 4.0, 'ko', ms=5)

# ISS
ax.annotate('', xy=(5, 1.8), xytext=(5, 3.0),
            arrowprops=dict(arrowstyle='->', color=COLOR_DESTAQUE, lw=2.5))
ax.text(5.5, 2.3, '$I_{SS}$', fontsize=13, color=COLOR_DESTAQUE, fontweight='bold')

# Ground
ax.plot([4.5, 5.5], [1.3, 1.3], 'k-', lw=2)
ax.plot([4.7, 5.3], [1.15, 1.15], 'k-', lw=1.5)
ax.plot([4.85, 5.15], [1.0, 1.0], 'k-', lw=1)
ax.plot([5, 5], [1.8, 1.3], 'k-', lw=1.5)

# Formula annotation
ax.text(5, 0.3, '$A_d = g_m (r_{o2} \| r_{o4})$', fontsize=14, ha='center',
        fontweight='bold', color=COLOR_PRINCIPAL,
        bbox=dict(boxstyle='round,pad=0.4', facecolor='white', edgecolor=COLOR_PRINCIPAL, lw=1.5))

plt.tight_layout()
plt.show()

### Comparacion carga pasiva vs carga activa

| Parametro | Carga pasiva ($R_D$) | Carga activa (espejo) |
|---|---|---|
| $A_d$ | $g_m R_D$ | $g_m (r_{o2} \| r_{o4})$ |
| Salida | Diferencial | Single-ended |
| $A_d$ tipico | 10 - 50 V/V | 200 - 2000 V/V |
| CMRR tipico | 40 - 60 dB | 80 - 120 dB |
| Consumo de area | Mayor (resistencias) | Menor (integrado) |

## 7. CMRR en frecuencia <a id="7-cmrr-frecuencia"></a>

La fuente de corriente $I_{SS}$ tiene una impedancia finita que disminuye con la frecuencia debido a capacidades parasitas. El CMRR no es constante:

$$\text{CMRR}(f) = \frac{A_d}{A_{cm}(f)} = 2 g_m |Z_{SS}(f)|$$

Donde $Z_{SS}(f) = R_{SS} \| \frac{1}{j 2\pi f C_{SS}}$ modela la capacidad parasita del nodo de cola.

A frecuencias altas, $|Z_{SS}|$ disminuye y el CMRR se degrada.

In [ ]:
# CMRR vs frecuencia
gm = 1.265e-3  # A/V (del calculo anterior)
RD = 10e3
RSS = 200e3  # buena fuente de corriente
CSS_vals = [0.5e-12, 2e-12, 10e-12]  # capacidad parasita del nodo cola

f = np.logspace(1, 9, 1000)  # 10 Hz a 1 GHz
w = 2 * np.pi * f

Ad_dB = 20 * np.log10(gm * RD)

fig, ax = plt.subplots(figsize=(10, 5))
colors_css = [COLOR_PRINCIPAL, COLOR_RECTA, COLOR_DESTAQUE]

for CSS, col in zip(CSS_vals, colors_css):
    Zss = RSS / (1 + 1j * w * CSS * RSS)
    Acm = -gm * RD / (1 + 2 * gm * Zss)
    CMRR = np.abs(-gm * RD / Acm)
    CMRR_dB = 20 * np.log10(CMRR)
    ax.semilogx(f, CMRR_dB, color=col, lw=2.5, label=f'$C_{{SS}}$ = {CSS*1e12:.1f} pF')

ax.axhline(Ad_dB, color='gray', ls=':', lw=1, label=f'$|A_d|$ = {Ad_dB:.1f} dB (limite)')
ax.set_xlabel('Frecuencia (Hz)', fontsize=12)
ax.set_ylabel('CMRR (dB)', fontsize=12)
ax.set_title('CMRR vs frecuencia para distintas capacidades parasitas $C_{SS}$',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.set_ylim(0, 120)
ax.set_xlim(10, 1e9)

# Annotate degradation
ax.annotate('Degradacion por\ncapacidad parasita', xy=(1e7, 40), fontsize=11,
            color=COLOR_RECTA, fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='white', edgecolor=COLOR_RECTA))

plt.tight_layout()
plt.show()

## 8. Ejercicios resueltos <a id="8-ejercicios"></a>

A continuacion se presentan 4 ejercicios que cubren los conceptos fundamentales del par diferencial.

### Ejercicio 1: Calculo basico de ganancias

Un par diferencial NMOS tiene $k_n = 1$ mA/V$^2$, $I_{SS} = 0.2$ mA, $R_D = 20$ k$\Omega$, $R_{SS} = 100$ k$\Omega$.

Calcular: $g_m$, $A_d$, $A_{cm}$, CMRR.

In [ ]:
# Ejercicio 1: Calculo basico
kn = 1e-3    # A/V^2
ISS = 0.2e-3  # A
RD = 20e3     # ohm
RSS = 100e3   # ohm

# Transconductancia
ID = ISS / 2  # cada transistor lleva ISS/2
Vov = np.sqrt(2 * ID / kn)
gm = 2 * ID / Vov  # equivalente a sqrt(2*kn*ID)

# Ganancias
Ad = gm * RD
Acm = gm * RD / (1 + 2*gm*RSS)
CMRR = Ad / Acm

print('=== Ejercicio 1: Par diferencial NMOS ===')
print(f'  ID = ISS/2 = {ID*1e3:.2f} mA')
print(f'  Vov = sqrt(2*ID/kn) = {Vov*1e3:.1f} mV')
print(f'  gm = 2*ID/Vov = {gm*1e3:.3f} mA/V')
print()
print(f'  Ad = gm*RD = {Ad:.1f} V/V = {20*np.log10(Ad):.1f} dB')
print(f'  Acm = gm*RD/(1+2*gm*RSS) = {Acm:.4f} V/V')
print(f'  CMRR = Ad/Acm = {CMRR:.0f} = {20*np.log10(CMRR):.1f} dB')

### Ejercicio 2: Rango de entrada lineal MOS

Para el par diferencial del Ej. 1, determinar:
- El rango de $v_{id}$ para operacion lineal (ambos transistores en saturacion)
- La tension diferencial de salida maxima

In [ ]:
# Ejercicio 2: Rango lineal
kn = 1e-3
ISS = 0.2e-3
RD = 20e3

Vov = np.sqrt(ISS / kn)  # sqrt(ISS/kn) para el calculo del rango
vid_max = Vov * np.sqrt(2)  # vid maximo para ambos en saturacion

# Tension de salida diferencial maxima
gm = np.sqrt(2 * kn * (ISS/2))
vod_swing = ISS * RD  # maximo swing completo (un transistor corta, otro lleva ISS)

print('=== Ejercicio 2: Rango lineal ===')
print(f'  Vov = sqrt(ISS/kn) = {Vov*1e3:.1f} mV')
print(f'  vid_max = Vov*sqrt(2) = +/-{vid_max*1e3:.1f} mV')
print(f'  (fuera de este rango, un transistor corta)')
print()
print(f'  Swing maximo de salida diferencial:')
print(f'  vod_max = ISS * RD = +/-{vod_swing:.1f} V')
print(f'  (limitado por la corriente de cola)')

### Ejercicio 3: Par diferencial con carga activa

Un par diferencial con espejo PMOS tiene $g_m = 1$ mA/V, $r_{o,n} = 200$ k$\Omega$, $r_{o,p} = 300$ k$\Omega$.

Calcular la ganancia diferencial y compararla con carga pasiva $R_D = 10$ k$\Omega$.

In [ ]:
# Ejercicio 3: Carga activa vs pasiva
gm = 1e-3      # A/V
ro_n = 200e3   # ohm (ro de NMOS)
ro_p = 300e3   # ohm (ro de PMOS)
RD = 10e3      # carga pasiva

# Carga pasiva
Ad_pasiva = gm * RD

# Carga activa (espejo): Av = gm * (ro_n || ro_p)
ro_par = (ro_n * ro_p) / (ro_n + ro_p)
Ad_activa = gm * ro_par

mejora = Ad_activa / Ad_pasiva

print('=== Ejercicio 3: Carga activa vs pasiva ===')
print(f'  gm = {gm*1e3:.1f} mA/V')
print(f'  ro_n = {ro_n/1e3:.0f} kohm')
print(f'  ro_p = {ro_p/1e3:.0f} kohm')
print(f'  ro_n || ro_p = {ro_par/1e3:.0f} kohm')
print()
print(f'  Con carga pasiva RD = {RD/1e3:.0f} kohm:')
print(f'    Ad = gm*RD = {Ad_pasiva:.0f} V/V = {20*np.log10(Ad_pasiva):.1f} dB')
print()
print(f'  Con carga activa (espejo PMOS):')
print(f'    Ad = gm*(ro_n||ro_p) = {Ad_activa:.0f} V/V = {20*np.log10(Ad_activa):.1f} dB')
print()
print(f'  Mejora: x{mejora:.0f} ({20*np.log10(mejora):.1f} dB mas)')

### Ejercicio 4: CMRR con fuente de corriente real

Un par diferencial tiene $g_m = 2$ mA/V, $R_D = 5$ k$\Omega$. La fuente de corriente de cola tiene $R_{SS} = 500$ k$\Omega$ y una capacidad parasita $C_{SS} = 1$ pF.

Calcular CMRR a DC y a $f = 10$ MHz.

In [ ]:
# Ejercicio 4: CMRR en frecuencia
gm = 2e-3    # A/V
RD = 5e3     # ohm
RSS = 500e3  # ohm
CSS = 1e-12  # F

# DC
Ad = gm * RD
Acm_dc = gm * RD / (1 + 2*gm*RSS)
CMRR_dc = Ad / Acm_dc

print('=== Ejercicio 4: CMRR en frecuencia ===')
print(f'  Ad = gm*RD = {Ad:.0f} V/V')
print()
print(f'--- A DC (f = 0) ---')
print(f'  Acm = {Acm_dc:.5f} V/V')
print(f'  CMRR = {CMRR_dc:.0f} = {20*np.log10(CMRR_dc):.1f} dB')
print()

# A 10 MHz
f = 10e6
w = 2 * np.pi * f
Zss = RSS / (1 + 1j * w * CSS * RSS)
Acm_f = gm * RD / (1 + 2*gm*Zss)
CMRR_f = abs(Ad / Acm_f)

fp = 1 / (2 * np.pi * RSS * CSS)
print(f'  Polo de la fuente de corriente: fp = 1/(2*pi*RSS*CSS) = {fp/1e3:.0f} kHz')
print()
print(f'--- A f = {f/1e6:.0f} MHz ---')
print(f'  |Zss| = {abs(Zss)/1e3:.1f} kohm (muy reducido)')
print(f'  |Acm| = {abs(Acm_f):.4f} V/V')
print(f'  CMRR = {CMRR_f:.0f} = {20*np.log10(CMRR_f):.1f} dB')
print()
print(f'  Degradacion: {20*np.log10(CMRR_dc) - 20*np.log10(CMRR_f):.1f} dB de perdida')

## 9. Catalogo de problemas tipo <a id="9-catalogo"></a>

Referencia rapida de los 5 tipos de problemas mas comunes sobre el par diferencial.

| # | Tipo de problema | Datos tipicos | Que calcular |
|---|---|---|---|
| 1 | **Calculo basico de ganancias** | $k_n$, $I_{SS}$, $R_D$, $R_{SS}$ | $g_m$, $A_d$, $A_{cm}$, CMRR |
| 2 | **Rango de entrada lineal** | $k_n$, $I_{SS}$ | $v_{id,max} = \sqrt{2I_{SS}/k_n}$ |
| 3 | **Carga activa (espejo)** | $g_m$, $r_{o,n}$, $r_{o,p}$ | $A_d = g_m(r_{o2}\|r_{o4})$ |
| 4 | **CMRR en frecuencia** | $R_{SS}$, $C_{SS}$, $f$ | $Z_{SS}(f)$, CMRR(f) |
| 5 | **Diseno para especificacion** | CMRR min, $A_d$ min | Elegir $R_{SS}$, $k_n$, $I_{SS}$ |

### Pasos clave por tipo

**Tipo 1 - Ganancias:** Calcular $g_m = \sqrt{2k_n I_{SS}}$, luego $A_d = g_m R_D$, $A_{cm} = g_m R_D/(1+2g_m R_{SS})$, $\text{CMRR} = A_d/A_{cm}$.

**Tipo 2 - Rango lineal:** $v_{id,max} = V_{OV}\sqrt{2}$ con $V_{OV} = \sqrt{I_{SS}/k_n}$. Para BJT, rango $\approx \pm 4V_T \approx \pm 104$ mV.

**Tipo 3 - Carga activa:** Identificar $r_o$ de cada transistor. $A_d = g_m(r_{o,NMOS} \| r_{o,PMOS})$. La salida es single-ended.

**Tipo 4 - CMRR(f):** Calcular polo $f_p = 1/(2\pi R_{SS} C_{SS})$. Para $f \gg f_p$: $|Z_{SS}| \approx 1/(2\pi f C_{SS})$ y CMRR cae.

**Tipo 5 - Diseno:** De la especificacion CMRR $\geq$ X, obtener $R_{SS} \geq \text{CMRR}/(2g_m)$. Iterar con $A_d$ requerida.

## 10. Resumen final <a id="10-resumen"></a>

### Formulas fundamentales del par diferencial

In [ ]:
# Tabla resumen visual
fig, ax = plt.subplots(figsize=(11, 7))
ax.axis('off')
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)

ax.text(5, 9.5, 'PAR DIFERENCIAL - RESUMEN DE FORMULAS', fontsize=16,
        ha='center', fontweight='bold', color=COLOR_PRINCIPAL)

# Definiciones
y = 8.5
ax.text(0.5, y, 'DEFINICIONES', fontsize=13, fontweight='bold', color=COLOR_RECTA)
y -= 0.6
formulas_def = [
    r'$v_{id} = v_{G1} - v_{G2}$       $v_{cm} = (v_{G1}+v_{G2})/2$',
    r'$g_m = \sqrt{2 k_n I_{SS}} = 2I_D / V_{OV}$       (con $I_D = I_{SS}/2$)',
]
for f in formulas_def:
    ax.text(0.8, y, f, fontsize=12, va='center')
    y -= 0.5

# Carga pasiva
y -= 0.3
ax.text(0.5, y, 'CARGA PASIVA ($R_D$)', fontsize=13, fontweight='bold', color=COLOR_RECTA)
y -= 0.6
formulas_pas = [
    r'$A_d = -g_m R_D$',
    r'$A_{cm} = -g_m R_D / (1 + 2g_m R_{SS}) \approx -R_D/(2R_{SS})$',
    r'$\mathrm{CMRR} = 1 + 2g_m R_{SS} \approx 2g_m R_{SS}$',
]
for f in formulas_pas:
    ax.text(0.8, y, f, fontsize=12, va='center')
    y -= 0.5

# Carga activa
y -= 0.3
ax.text(0.5, y, 'CARGA ACTIVA (espejo de corriente)', fontsize=13, fontweight='bold', color=COLOR_RECTA)
y -= 0.6
formulas_act = [
    r'$A_d = g_m (r_{o2} \| r_{o4})$       (salida single-ended)',
    r'CMRR mucho mayor (depende de matching)',
]
for f in formulas_act:
    ax.text(0.8, y, f, fontsize=12, va='center')
    y -= 0.5

# Gran senal
y -= 0.3
ax.text(0.5, y, 'GRAN SENAL', fontsize=13, fontweight='bold', color=COLOR_RECTA)
y -= 0.6
formulas_gs = [
    r'MOS: rango lineal $|v_{id}| \leq V_{OV}\sqrt{2}$ con $V_{OV} = \sqrt{I_{SS}/k_n}$',
    r'BJT: rango lineal $|v_{id}| \leq 4V_T \approx 104$ mV',
]
for f in formulas_gs:
    ax.text(0.8, y, f, fontsize=12, va='center')
    y -= 0.5

# Frecuencia
y -= 0.3
ax.text(0.5, y, 'CMRR EN FRECUENCIA', fontsize=13, fontweight='bold', color=COLOR_RECTA)
y -= 0.6
ax.text(0.8, y, r'$\mathrm{CMRR}(f) = 2g_m |Z_{SS}(f)|$, polo en $f_p = 1/(2\pi R_{SS} C_{SS})$', fontsize=12)

plt.tight_layout()
plt.show()

### Puntos clave para recordar

1. El par diferencial amplifica $v_{id}$ y rechaza $v_{cm}$ gracias a la simetria del circuito.
2. La fuente de corriente de cola $I_{SS}$ es critica: su impedancia determina el CMRR.
3. Con carga pasiva ($R_D$): ganancia moderada, salida diferencial.
4. Con carga activa (espejo): ganancia alta, salida single-ended, base del op-amp.
5. El rango lineal MOS es mayor que el BJT, pero el BJT tiene mayor $g_m$ para la misma corriente.
6. A frecuencias altas, el CMRR se degrada por capacidades parasitas en el nodo de cola.